# Polarisation
On calcule une mesure de polarisation. Pour cela on va calculer la proportion de messages violents en réponse à des posts ou commentaires du bord opposé. De même, pour l'inverse (messages non violent pour le même bord, et potentiellement neutre).  

Ensuite, analyser l'évolution de cette mesure : est ce que les gens sur Reddit le sgens sont de plus en plus violents dans leurs messages ? Envers le bord opposé particulièrement ? Y a-t-il des périodes particulièrement houleuses ?

A adapter aux noms corrects des colones concernant les bords politiques et les réponses/commentaires.

## Calcul de la mesure de la polarisation

In [1]:
pip install hmmlearn


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM

In [ ]:
# importer les données
df_1 = pd.read_csv("flat_english_left_right_interactions.csv")
df_2 = pd.read_csv("flat_french_left_right_interactions.csv")

In [ ]:
df = pd.concat([df_1, df_2])

,text,left_right
0,my parents are kurdish immigrants that moved t...,Left
1,So I'm probably gonna get down voted into obli...,Right
2,Finishing up iberia because that dumbass franc...,Left
3,How do I increase control/proximity to capital...,Left
4,Everybody saw complaints about France being a ...,Right
...,...,...
41999,"BU is terrible with anything race related, eve...",Right
42000,Newsweek stirring up shit. They don't care abo...,Right
42001,This subreddit is essentially a Russian run tr...,Right
42002,A country in Africa this week announced the de...,Left


In [10]:
df_tf = pd.read_csv("../toxicity/political_interactions_toxic.csv")
df_tf = df_tf.drop(columns=["Unnamed: 0"])

In [11]:
df_tf

,text,toxicity_level
0,my parents are kurdish immigrants that moved t...,0.016723
1,On voit partout sur les réseaux des publicatio...,0.003801
2,"Les coiffeurs ne figurant pas, à ma connaissan...",0.243365
3,L'Établissement Français du Sang n'a pas inter...,0.007407
4,"On peut accuser internet, les films, les jeux ...",0.019245
...,...,...
164286,Tu confonds public et privé.Un fois de plus ce...,0.003015
164287,Je ne pense pas que l'idée soit de le défendre...,0.017053
164288,"Non, ils sont vraiment comme ça.J'en ai vu qui...",0.005502
164289,Il avait son fan club dès le départ juste parc...,0.096742


In [ ]:
# Rajouter une colone pour indiquer si c'est une réponse au camp opposé - A voir si on affine pour séparer les X / X radical
df["opposition"] = (
    df["bord_politique"] != df["bord_original"]
).astype(int)

In [ ]:
# En premier, très général, toxicité moyenne en fonction de la réponse au camp opposé, ou 
df["date"] = pd.to_datetime(df["date"])

evolution = (
    df.groupby([
        pd.Grouper(key="date", freq="M"),
        "opposition"
    ])["toxicity_level"]
    .mean()
    .reset_index()
)

In [ ]:
# En regroupant par bord politique
df["date"] = pd.to_datetime(df["date"])

evolution = (
    df.groupby([
        pd.Grouper(key="date", freq="M"),
        "opposition",
        "bord_politique"
    ])["toxicity_level"]
    .mean()
    .reset_index()
)

evolution.columns = [
    "meme_camp",
    "camp_oppose"
]

print(evolution)

In [ ]:
# Par utilisateur ? Je sais pas si c'est pas trop fin
df["date"] = pd.to_datetime(df["date"])

user_stat = (
    df.groupby([
        pd.Grouper(key="date", freq="M"),
        "opposition",
        "user"
    ])["toxicity_level"]
    .mean()
    .reset_index()
)

user_stat.columns = [
    "meme_camp",
    "camp_oppose"
]

user_stats["difference"] = (
    user_stats["camp_oppose"]
    - user_stats["meme_camp"]
)

print(user_stats.sort_values("difference", ascending=False))

## Analyse de l'évolution

In [ ]:
# Petit plot, prendre soit evolution, par bord, soit user stat par utilisateur, mais surement pas assez lisible
for val in [0, 1]:

    subset = evolution[
        evolution["reponse_opposition"] == val
    ]

    plt.plot(
        subset["date"],
        subset["toxicite"],
        label=f"Opposition={val}"
    )

plt.legend()
plt.ylabel("Toxicité moyenne")
plt.xlabel("Date")
plt.show()

In [ ]:
# Evolution de l'écart
pivot = evolution.pivot(
    index="date",
    columns="reponse_opposition",
    values="toxicite"
)

pivot["ecart"] = pivot[1] - pivot[0]

pivot["ecart"].plot()
plt.ylabel("Ecart de toxicité, entre bord opposé et bord allié")
plt.show()

### HMM pour essayer de séparer éventuellement des périodes plus calmes, ou plus houleuses

In [ ]:
X = pivot["ecart"].dropna().values.reshape(-1, 1)

model = GaussianHMM(
    n_components=2,
    covariance_type="full",
    random_state=42
)

model.fit(X)

states = model.predict(X)

pivot = pivot.dropna()
pivot["periodes"] = states